# 01 — Ingest

Fetch the raw source and land it in `data/raw/` **untouched**.

**Source:** Box Office Mojo — *Top Lifetime Adjusted Grosses* (domestic). The
`?adjust_gross_to=2022` query param makes the page return each film's
inflation-adjusted gross (in 2022 dollars) alongside its nominal gross,
estimated tickets sold, and release year. With that param the page is static
(no JS needed).

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config, fetch_html

cfg = load_config('config.yaml')
src = cfg['sources']['bom_adjusted']
url = src['url']
print('Source URL:', url)

## Fetch and cache the raw HTML
Save the page to `data/raw/` so the raw source is preserved and re-runs are
offline.

In [ ]:
raw_path = Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html'
raw_path.parent.mkdir(parents=True, exist_ok=True)

if raw_path.exists():
    print('Using cached raw HTML:', raw_path)
    html = raw_path.read_text(encoding='utf-8')
else:
    html = fetch_html(url)
    raw_path.write_text(html, encoding='utf-8')
    print('Saved raw HTML ->', raw_path)

print(f'{len(html):,} bytes')

## Parse the chart table
The page has a single table: Rank, Title, Adj. Lifetime Gross, Lifetime Gross,
Est. Num Tickets, Year.

In [ ]:
raw = pd.read_html(StringIO(html))[0]
print(raw.shape)
raw.head(10)

## Also fetch the WORLDWIDE chart
The adjusted chart above is **domestic only** (U.S. & Canada). To compare
domestic vs. international we also pull Box Office Mojo's worldwide chart,
which splits each film into worldwide / domestic / foreign gross.

In [ ]:
ww_url = cfg['sources']['bom_worldwide']['url']
ww_path = Path(cfg['paths']['data_raw']) / 'bom_ww_top_lifetime.html'
if ww_path.exists():
    ww_html = ww_path.read_text(encoding='utf-8')
else:
    ww_html = fetch_html(ww_url)
    ww_path.write_text(ww_html, encoding='utf-8')
ww_raw = pd.read_html(StringIO(ww_html))[0]
print(ww_raw.shape)
ww_raw.head(5)

---
**Next:** `02-clean.ipynb` loads these into DuckDB, cleans them, and enriches
with genre from TMDB.

Nothing here modified data — the raw HTML in `data/raw/` is untouched.